# **A Hybrid model implementation**

## By using Parameterized Rx , Ry gates and entanglement

In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 89.0 MB/s eta 0:00:00


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset
import pennylane as qml
from tqdm import tqdm

In [ ]:
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)  # smaller batch
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
# circuit setup
n_qubits = 4
n_layers = 5
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def improved_quantum_circuit(inputs, weights):
    for i in range(n_layers):
        # Data re-uploading
        qml.AngleEmbedding(inputs, wires=range(n_qubits))

        # Parameterized rotations
        for wire in range(n_qubits):
            qml.RY(weights[i, wire, 0], wires=wire)
            qml.RZ(weights[i, wire, 1], wires=wire)

        # Circular entangling CNOTs
        for wire in range(n_qubits):
            qml.CNOT(wires=[wire, (wire + 1) % n_qubits])

    # Measuring PauliZ and PauliX on each qubit
    return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)] + \
           [qml.expval(qml.PauliX(w)) for w in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits, 2)}
quantum_layer = qml.qnn.TorchLayer(improved_quantum_circuit, weight_shapes)

# Hybrid Model Definition
class HybridCNNQNN(nn.Module):
    def __init__(self):
        super(HybridCNNQNN, self).__init__()
        self.fc1 = nn.Linear(28 * 28, n_qubits)
        self.qnn = quantum_layer
        self.fc2 = nn.Linear(n_qubits * 2, 32)  # Note: quantum output size doubled (Z and X)
        self.fc3 = nn.Linear(32, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        x = self.qnn(x)
        x = torch.relu(x)
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Setup device, model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_basic = HybridCNNQNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_basic.parameters(), lr=0.001)

# Loading the pretrained weights ( use the file same as in github)
model_basic.load_state_dict(torch.load('models/model_using_RxAndRy.pth'))
# Evaluation
model_basic.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_basic(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Accuracy: 64.07%


## using quanvalution


In [ ]:
# Quantum circuit for Quanvolution (2x2 patch -> 4 qubits)
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quanv_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (1, n_qubits, 3)}
quanv_layer = qml.qnn.TorchLayer(quanv_circuit, weight_shapes)

class QuanvolutionLayer(nn.Module):
    def __init__(self, kernel_size=2, in_channels=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.in_channels = in_channels
        self.quanv = quanv_layer

    def forward(self, x):
        batch_size, channels, height, width = x.size()
        patches_list = []
        for c in range(channels):
            patches = x[:, c:c+1, :, :].unfold(2, self.kernel_size, 1).unfold(3, self.kernel_size, 1)
            patches = patches.contiguous().view(batch_size, -1, self.kernel_size * self.kernel_size)
            patches_list.append(patches)
        patches = torch.cat(patches_list, dim=1)
        quanv_out = self.quanv(patches.reshape(-1, self.kernel_size * self.kernel_size))
        output_dim = quanv_out.size(1)
        out_size = height - self.kernel_size + 1
        quanv_out = quanv_out.view(batch_size, -1, out_size, out_size)
        return quanv_out

class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.quanv = QuanvolutionLayer(kernel_size=2, in_channels=1)

        self.conv_layers = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self._to_linear = None
        self._get_conv_output()

        self.fc1 = nn.Linear(self._to_linear, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 10)

    def _get_conv_output(self):
        with torch.no_grad():
            x = torch.randn(1, 4, 27, 27)  # Quanv layer output dims
            x = self.conv_layers(x)
            self._to_linear = x.view(1, -1).size(1)

    def forward(self, x):
        x = self.quanv(x)
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.relu(self.fc3(x))
        x = self.out(x)
        return x


# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_quanv = HybridModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_quanv.parameters(), lr=0.001)

# Using pretrained weights ( use the same filename as in github )

model_quanv.load_state_dict(torch.load('models/hybrid_model_state.pth'))
# Evaluation
model_quanv.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_quanv(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Accuracy: 76.02%
